# Scientific Challenge: ANNNI Phase Detection with VQE

**Team:** WestQuantOpen  
**Challenge:** Q-SITE 2026 Open Challenge  

---

## Overview

This notebook presents our solution to the Scientific Challenge: detecting quantum phases of the ANNNI (Axial Next-Nearest-Neighbor Ising) model using variational quantum eigensolvers, and quantifying the effect of gate noise on phase detection.

### Three Critical Fixes

1. **Structure factor correction** — Fixed a systematic -1.0 offset from missing diagonal terms
2. **Antiphase wavevector correction** — q* = π/2 (period 4: ↑↑↓↓), not π
3. **Phase-adapted reference-state HVA** — U_HVA(θ)|ψ_ref⟩ instead of U_HVA(θ)|0^N⟩

### Key Results

- Classifier accuracy: 25.85% → 70.1% (100% on interior points)
- Antiphase VQE: ΔE 2.16 → 0.02
- Challenge-compliant noisy pipeline with quantitative degradation data

## 1. The ANNNI Model

The Axial Next-Nearest-Neighbor Ising (ANNNI) model is a 1D quantum spin chain with:

$$H = -\sum_i \left[\kappa \, \sigma^Z_i \sigma^Z_{i+2} + \sigma^Z_i \sigma^Z_{i+1} + h \, \sigma^X_i\right]$$

where:
- $\kappa$ controls next-nearest-neighbor (frustrating) coupling
- $h$ is the transverse field strength
- The competition between NN and NNN interactions creates rich phase structure

### Phase Diagram

| Phase | Region | Order Parameter |
|-------|--------|-----------------|
| **Ferromagnetic** | Low κ, low h | S(0) large, q* ≈ 0 |
| **Antiphase** | High κ, low h | S(π/2) large, q* ≈ π/2 |
| **Paramagnetic** | High h | ⟨X⟩ large, S(q) flat |
| **Floating** | Intermediate | q* incommensurate |

**Key insight:** The antiphase pattern ↑↑↓↓ has period 4, so its characteristic wavevector is q* = π/2, **not** π.

## 2. Structure Factor Fix (Critical)

### The Bug

The reference structure factor computation was **missing diagonal terms** (i=j), producing S(q) - 1 instead of S(q). This caused:
- Impossible negative S(0) values (S(q) ≥ 0 is guaranteed by the correct formula)
- Incorrect q* values
- Complete classifier failure on antiphase states

### The Correct Formula

$$S(q) = \frac{1}{N} \sum_{i,j} e^{iq(i-j)} \langle Z_i Z_j \rangle = \frac{1}{N} v(q)^\dagger C \, v(q)$$

where $v_j(q) = e^{iqj}$ and $C_{ij} = \langle Z_i Z_j \rangle$.

This equals $\frac{1}{N} \langle |\sum_j e^{iqj} Z_j|^2 \rangle \geq 0$, guaranteeing non-negativity.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Corrected structure factor implementation
def compute_structure_factor(zz_matrix, n_qubits, q_values):
    """Compute S(q) = (1/N) * v(q)^† C v(q) where C_ij = <Z_i Z_j>."""
    s_q = np.zeros(len(q_values))
    for qi, q in enumerate(q_values):
        total = 0.0
        for i in range(n_qubits):
            for j in range(n_qubits):
                total += np.cos(q * (i - j)) * zz_matrix[i, j]
        s_q[qi] = total / n_qubits
    return s_q

# Verify on analytic states
n = 8
q = np.linspace(0, 2*np.pi, 256, endpoint=False)

# Ferromagnetic |00000000⟩: <Z_i Z_j> = 1 for all i,j
zz_ferro = np.ones((n, n))
s_ferro = compute_structure_factor(zz_ferro, n, q)

# Antiphase |00110011⟩: Z = [+1,+1,-1,-1,+1,+1,-1,-1]
ap_z = np.array([1, 1, -1, -1, 1, 1, -1, -1])
zz_anti = np.outer(ap_z, ap_z)
s_anti = compute_structure_factor(zz_anti, n, q)

# Paramagnetic |+⟩^8: <Z_i Z_j> = δ_ij
zz_para = np.eye(n)
s_para = compute_structure_factor(zz_para, n, q)

print("=== Structure Factor Verification ===")
print(f"Ferromagnetic:  S(0)={s_ferro[0]:.3f}, S(π/2)={s_ferro[len(q)//4]:.3f}, S(π)={s_ferro[len(q)//2]:.3f}")
print(f"Antiphase:      S(0)={s_anti[0]:.3f}, S(π/2)={s_anti[len(q)//4]:.3f}, S(π)={s_anti[len(q)//2]:.3f}")
print(f"Paramagnetic:   S(0)={s_para[0]:.3f}, S(π/2)={s_para[len(q)//4]:.3f}, S(π)={s_para[len(q)//2]:.3f}")
print(f"\nAll S(q) ≥ 0: {np.all(s_ferro >= -1e-10) and np.all(s_anti >= -1e-10) and np.all(s_para >= -1e-10)}")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].plot(q, s_ferro, linewidth=2, color='#e74c3c')
axes[0].set_title('Ferromagnetic |00000000⟩', fontsize=12, fontweight='bold')
axes[0].set_xlabel('q'); axes[0].set_ylabel('S(q)')
axes[0].axvline(x=0, color='gray', linestyle='--', alpha=0.5)
axes[0].grid(alpha=0.3)

axes[1].plot(q, s_anti, linewidth=2, color='#3498db')
axes[1].set_title('Antiphase |00110011⟩', fontsize=12, fontweight='bold')
axes[1].set_xlabel('q'); axes[1].set_ylabel('S(q)')
axes[1].axvline(x=np.pi/2, color='gray', linestyle='--', alpha=0.5, label='q=π/2')
axes[1].legend()
axes[1].grid(alpha=0.3)

axes[2].plot(q, s_para, linewidth=2, color='#2ecc71')
axes[2].set_title('Paramagnetic |+⟩^8', fontsize=12, fontweight='bold')
axes[2].set_xlabel('q'); axes[2].set_ylabel('S(q)')
axes[2].grid(alpha=0.3)

plt.suptitle('Structure Factor S(q) for Analytic States', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../presentations/images/structure_factor.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Classifier Fix

### The Bug

The classifier was checking S(π) for antiphase. ANNNI antiphase has period 4 (↑↑↓↓), so the correct wavevector is **q* = π/2**, not π.

### Corrected Classification Rules

| Phase | Condition |
|-------|-----------|
| Ferromagnetic | S(0) > 3.0 |
| Antiphase | S(π/2) > 1.5 |
| Paramagnetic | ⟨X⟩ > 0.85 |
| Floating | q* incommensurate (not 0 or π/2) |

In [ ]:
import json

# Load classifier results
with open('../results/classifier_sanity_check_corrected.json') as f:
    classifier = json.load(f)

print("=== Classifier Accuracy ===")
print(f"Overall: {classifier['accuracy']:.1%} ({classifier['correct']}/{classifier['total']})")
print()
for phase, stats in classifier['per_phase'].items():
    acc = stats['correct']/stats['total'] if stats['total'] > 0 else 0
    print(f"  {phase}: {acc:.1%} ({stats['correct']}/{stats['total']})")

# Plot comparison
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(10, 6))
phases = ['Ferro', 'Antiphase', 'Paramag', 'Overall', 'Interior']
before = [68.5, 0.0, 0.0, 25.85, 0.0]
after = [92.3, 93.3, 81.3, 70.1, 100.0]

x = np.arange(len(phases))
width = 0.35
ax.bar(x - width/2, before, width, label='Before Fix', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, after, width, label='After Fix', color='#2ecc71', alpha=0.8)

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Classifier Accuracy: Before vs After Fix', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(phases, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 110)

plt.tight_layout()
plt.savefig('../presentations/images/classifier_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Phase-Adapted Reference-State HVA (Breakthrough)

### The Problem

Standard VQE uses U_HVA(θ)|0^N⟩. This works for ferromagnetic states (ΔE=0.003) but fails for antiphase (ΔE=2.16) because the optimizer cannot find the correct variational basin from random parameters.

### Reference-State Sanity Test

We tested whether simple product states already have low ΔE at phase-specific points:

| Point | Reference State | ΔE | Verdict |
|-------|----------------|-----|---------|
| κ=0.2, h=0.2 | |00000000⟩ | 0.101 | LOW |
| κ=0.8, h=0.3 | |00110011⟩ | 0.241 | LOW |
| κ=0.5, h=1.5 | |+⟩^8 | 1.194 | MEDIUM |

**Key finding:** The antiphase reference state has ΔE=0.24 — LOW! The HVA L3 **can** represent the state, but the optimizer can't find it. **The problem is state preparation, not ansatz expressivity.**

### The Solution: Phase-Adapted HVA

Instead of U_HVA(θ)|0^N⟩, use U_HVA(θ)|ψ_ref⟩ where:
- Ferromagnetic: |ψ_ref⟩ = |00000000⟩
- Antiphase: |ψ_ref⟩ = |00110011⟩
- Paramagnetic: |ψ_ref⟩ = |+⟩^⊗N

The winning branch (lowest energy) provides an **independent phase signal**.

In [ ]:
# Load and display phase-adapted VQE results
with open('../results/phase_adapted_vqe_results.json') as f:
    vqe_results = json.load(f)

print("=== Phase-Adapted VQE Results ===\n")
print(f"{'Point':>20} {'ΔE':>10} {'Branch':>12} {'Phase':>15} {'Status':>8}")
print("-" * 70)
for r in vqe_results:
    status = '✓' if abs(r['delta_E']) < 0.1 else ('~' if abs(r['delta_E']) < 0.5 else '✗')
    print(f"  κ={r['kappa']:.1f} h={r['h']:.1f}    {r['delta_E']:>10.4f} {r['best_branch']:>12} {r['predicted_phase']:>15} {status:>8}")
    print(f"    S(0)={r['s0']:.3f}  S(π/2)={r['s_pi2']:.3f}  q*={r['q_star']:.3f}  <X>={r['x_mean']:.3f}")
    for bn, br in r['branch_results'].items():
        print(f"    {bn}: ΔE={br['delta_E']:.4f}")
    print()

# Plot VQE comparison
fig, ax = plt.subplots(figsize=(10, 6))
points = ['Ferro\nκ=0.2,h=0.2', 'Antiphase\nκ=0.8,h=0.3', 'Paramag\nκ=0.5,h=1.5']
old_vqe = [0.002, 2.162, 1.858]
new_vqe = [0.003, 0.021, 0.042]

x = np.arange(len(points))
width = 0.35
ax.bar(x - width/2, old_vqe, width, label='Standard HVA', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, new_vqe, width, label='Phase-Adapted HVA', color='#2ecc71', alpha=0.8)

ax.set_ylabel('ΔE (VQE - Exact)', fontsize=12)
ax.set_title('VQE Energy Accuracy: Standard vs Phase-Adapted', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(points)
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_yscale('log')

plt.tight_layout()
plt.savefig('../presentations/images/vqe_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Challenge-Compliant Noisy Pipeline

### Pipeline

1. Build HVA L3 circuit with phase-adapted reference state
2. Apply depolarizing channel after each layer (challenge-compliant noise model)
3. Measure S(q) and classify phase
4. Compare across noise levels p = 0, 0.01, 0.05

### Key Question

**Which type of quantum order is most noise-sensitive?**

In [ ]:
# Load noisy pipeline results
with open('../results/noisy_phase_adapted_fixed.json') as f:
    noisy_results = json.load(f)

print("=== Noise Degradation Results ===\n")
for name, r in noisy_results.items():
    print(f"--- {name} (branch={r['branch']}, ΔE={r['delta_E']:.4f}) ---")
    for p in ['0.0', '0.01', '0.05']:
        cls = r['noise'][p]
        print(f"  p={p}: {cls['phase']:>15}  S(0)={cls['s0']:.3f}  S(π/2)={cls['s_pi2']:.3f}  q*={cls['q_star']:.3f}  <X>={cls['x_mean']:.3f}")
    print()

# Plot noise degradation
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
noise_levels = [0.0, 0.01, 0.05]

# Ferro
ferro_s0 = [noisy_results['ferro_0.2_0.2']['noise']['0.0']['s0'],
            noisy_results['ferro_0.2_0.2']['noise']['0.01']['s0'],
            noisy_results['ferro_0.2_0.2']['noise']['0.05']['s0']]
axes[0].plot(noise_levels, ferro_s0, 'o-', linewidth=2, markersize=8, color='#e74c3c')
axes[0].set_title('Ferromagnetic S(0)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Noise p'); axes[0].set_ylabel('S(0)')
axes[0].grid(alpha=0.3)

# Antiphase
anti_s = [noisy_results['anti_0.8_0.3']['noise']['0.0']['s_pi2'],
          noisy_results['anti_0.8_0.3']['noise']['0.01']['s_pi2'],
          noisy_results['anti_0.8_0.3']['noise']['0.05']['s_pi2']]
axes[1].plot(noise_levels, anti_s, 's-', linewidth=2, markersize=8, color='#3498db')
axes[1].set_title('Antiphase S(π/2)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Noise p'); axes[1].set_ylabel('S(π/2)')
axes[1].grid(alpha=0.3)

# Paramag
para_x = [noisy_results['para_0.5_1.5']['noise']['0.0']['x_mean'],
          noisy_results['para_0.5_1.5']['noise']['0.01']['x_mean'],
          noisy_results['para_0.5_1.5']['noise']['0.05']['x_mean']]
axes[2].plot(noise_levels, para_x, '^-', linewidth=2, markersize=8, color='#2ecc71')
axes[2].set_title('Paramagnetic ⟨X⟩', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Noise p'); axes[2].set_ylabel('⟨X⟩')
axes[2].grid(alpha=0.3)

plt.suptitle('Noise Degradation by Phase (p=0, 0.01, 0.05)', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../presentations/images/noise_degradation.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary
print("=== Noise Degradation Summary (p=0.05) ===")
for name, r in noisy_results.items():
    p0 = r['noise']['0.0']
    p5 = r['noise']['0.05']
    stable = p0['phase'] == p5['phase']
    if 'ferro' in name:
        ds = p0['s0'] - p5['s0']
        pct = ds/p0['s0']*100
        print(f"  Ferro:    ΔS(0)={ds:.3f} ({pct:.1f}%)  phase_stable={stable}")
    elif 'anti' in name:
        ds = p0['s_pi2'] - p5['s_pi2']
        pct = ds/p0['s_pi2']*100
        print(f"  Anti:     ΔS(π/2)={ds:.3f} ({pct:.1f}%)  phase_stable={stable}")
    elif 'para' in name:
        ds = p0['x_mean'] - p5['x_mean']
        pct = ds/p0['x_mean']*100
        print(f"  Paramag:  Δ⟨X⟩={ds:.3f} ({pct:.1f}%)  phase_stable={stable}")

## 6. Key Findings

1. **Structure factor implementation had a systematic -1.0 offset** from missing diagonal terms — this caused impossible negative S(0) values and complete classifier failure
2. **Antiphase wavevector is π/2** (not π) for ANNNI period-4 (↑↑↓↓) ordering — this was the root cause of 0% antiphase accuracy
3. **Interior phases are easy to classify** but boundaries are hard — this is expected for finite-size (N=8) systems
4. **Shallow HVA L3 works nearly exactly in the ferro region** (ΔE=0.003) but fails dramatically in frustrated regimes without phase-adapted initialization
5. **Phase-adapted reference-state HVA solves all phases** (ΔE < 0.12) — the problem was state preparation, not ansatz expressivity
6. **Ferromagnetic order is most noise-sensitive** (29.7% S(0) degradation at p=0.05), followed by antiphase (25.2%), then paramagnetic (18.8%)
7. **The winning variational branch provides an independent phase signal** — antiphase branch wins at antiphase points, paramag branch at paramag points

## 7. Research Narrative

The project evolved from "map the ANNNI phase diagram" to a richer research question:

> **"How do representation choice and gate noise jointly affect quantum phase detection?"**

This is more interesting than simply showing three color maps — it reveals that the interplay between ansatz initialization, physical observables, and noise creates a multi-dimensional phase detection problem.